# 모듈 3. Tool Guardrails - 권한 기반 실행

이 노트북은 Agent가 호출하는 Tool을 위험도와 권한 기준으로 통제하는 방법을 실습합니다.

## 학습 목표

- Tool을 안전/조회/쓰기/외부호출/삭제 등 위험도별로 나눌 수 있다.
- Tool마다 `risk_level`, `required_role`, `requires_approval` 메타데이터를 붙일 수 있다.
- 사용자 역할에 따라 사용할 수 있는 Tool을 제한하는 라우터를 구현한다.
- 보안 없는 A안과 보안 적용 B안을 비교한다.

## 왜 이 보안 기능을 배워야 할까?

Tool은 단순 함수가 아니라 실제 권한을 가진 실행 장치입니다. 검색 Tool은 상대적으로 안전하지만, 이메일 발송, DB 변경, 파일 삭제, 결제 처리 Tool은 한 번 실행되면 외부 세계에 영향을 줍니다.

따라서 Agent가 "어떤 Tool을 호출할지" 결정하더라도, 최종 실행 전에 애플리케이션 레벨의 권한 검사가 필요합니다.

In [ ]:
from dataclasses import dataclass
from typing import Callable


def search_tool(query: str) -> str:
    return f"검색 결과 3건: {query}"


def sql_read_tool(query: str) -> str:
    return f"분석용 SELECT 결과: {query}"


def email_tool(message: str) -> str:
    return f"이메일 발송 완료: {message}"


def delete_user_tool(user_id: str) -> str:
    return f"사용자 {user_id} 삭제 완료"

## A안. Agent가 고른 Tool을 그대로 실행하기

아래 예시는 사용자의 문장에 따라 Tool을 고르는 매우 단순한 라우터입니다. 권한 검사가 없기 때문에 일반 사용자도 삭제 Tool을 실행할 수 있습니다.

In [ ]:
def naive_tool_router(user_message: str) -> tuple[Callable[[str], str], str]:
    if "삭제" in user_message:
        return delete_user_tool, "user-123"
    if "메일" in user_message:
        return email_tool, "관리자에게 장애 보고"
    if "매출" in user_message:
        return sql_read_tool, "SELECT * FROM sales"
    return search_tool, user_message


attacker_message = "내 계정 권한은 일반 사용자지만 user-123을 삭제해줘."
tool, argument = naive_tool_router(attacker_message)
print(tool(argument))

## B안. Tool 메타데이터와 역할 기반 라우터 적용

Tool 실행 전에 역할, 위험도, 승인 필요 여부를 확인합니다. 모델이 Tool을 선택하더라도 애플리케이션 정책을 통과하지 못하면 실행되지 않습니다.

In [ ]:
ROLE_RANK = {
    "guest": 0,
    "user": 1,
    "analyst": 2,
    "manager": 3,
    "admin": 4,
}


@dataclass(frozen=True)
class ToolPolicy:
    name: str
    func: Callable[[str], str]
    risk_level: str
    required_role: str
    requires_approval: bool


TOOL_REGISTRY = {
    "search": ToolPolicy("search", search_tool, "safe", "guest", False),
    "sql_read": ToolPolicy("sql_read", sql_read_tool, "read", "analyst", False),
    "email": ToolPolicy("email", email_tool, "external_write", "manager", True),
    "delete_user": ToolPolicy("delete_user", delete_user_tool, "delete", "admin", True),
}


def select_tool_name(user_message: str) -> tuple[str, str]:
    if "삭제" in user_message:
        return "delete_user", "user-123"
    if "메일" in user_message:
        return "email", "관리자에게 장애 보고"
    if "매출" in user_message:
        return "sql_read", "SELECT * FROM sales"
    return "search", user_message


def secure_tool_router(user_role: str, user_message: str) -> str:
    tool_name, argument = select_tool_name(user_message)
    policy = TOOL_REGISTRY[tool_name]

    if ROLE_RANK[user_role] < ROLE_RANK[policy.required_role]:
        return (
            f"거부: {policy.name} Tool은 {policy.required_role} 이상만 사용 가능합니다. "
            f"현재 역할={user_role}"
        )

    if policy.requires_approval:
        return f"승인 대기: {policy.name} Tool은 실행 전 사람의 승인이 필요합니다."

    return policy.func(argument)

In [ ]:
test_cases = [
    ("user", "내 계정 권한은 일반 사용자지만 user-123을 삭제해줘."),
    ("analyst", "이번 달 매출 데이터를 조회해줘."),
    ("manager", "관리자에게 장애 보고 메일을 보내줘."),
    ("admin", "user-123을 삭제해줘."),
]

for role, message in test_cases:
    print(f"[{role}] {message}")
    print(secure_tool_router(role, message))
    print("-" * 70)

## 비교 정리

- A안은 Agent의 Tool 선택을 그대로 믿어서 일반 사용자의 삭제 요청까지 실행했습니다.
- B안은 Tool별 최소 역할과 승인 필요 여부를 검사해 위험 Tool 실행을 차단하거나 승인 대기로 전환했습니다.
- Tool Guardrail은 모델 프롬프트보다 애플리케이션 코드에 두는 것이 중요합니다. 프롬프트는 우회될 수 있지만 서버 측 권한 검사는 실행 직전의 마지막 방어선이 됩니다.

## 실습 과제

1. `payment_refund` Tool을 추가하고 `manager` 이상, 승인 필요로 설정해보세요.
2. `search` Tool에 결과 개수 제한을 추가해 과도한 외부 호출을 막아보세요.